# Karpathy's micrograd `demo.ipynb`, ported to SKaiNET

Faithful port of Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd) `demo.ipynb` to SKaiNET's public DSLs:

- **Data**: idiomatic Kotlin `MoonPoint` records + the SKaiNET data DSL — `data<FP32, Float>(ctx) { tensor { shape(n, c) { fromList(...) } } }` — for in-memory tensor construction. Z-score statistics are fitted on the train split and applied to both splits.
- **Model**: `sequential<FP32, Float>(ctx) { input(2); dense(16); activation { it.tanh() }; ... }` — the NN DSL.
- **Training**: `training<FP32, Float> { model { } loss { MSELoss() } optimizer { sgd(lr = 0.1) } }` — the training DSL.
- **Visualisation**: Karpathy's classic decision-boundary plot — Kandy `tiles` over a meshgrid forward pass, with the training points overlaid as `points`.

Mirrors the SKaiNET 0.26.0 reference port in `skainet-lang-models/.../MicrogradMoonsDemoTest.kt`.

In [ ]:
@file:DependsOn("sk.ainet.app:kotlin-notebook:0.29.1")

In [ ]:
%use kandy
%use dataframe

## Imports and hyperparameters

`FP32` and `tanh` must be imported explicitly — Kotlin Jupyter's wildcard imports don't pick up types used as reified type-arguments (`<FP32, Float>`), and the `it.tanh()` call inside `activation { }` needs the extension in scope.

In [ ]:
import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.context.Phase
import sk.ainet.context.data
import sk.ainet.lang.graph.DefaultGradientTape
import sk.ainet.lang.graph.DefaultGraphExecutionContext
import sk.ainet.lang.nn.dsl.sequential
import sk.ainet.lang.nn.dsl.training
import sk.ainet.lang.nn.loss.MSELoss
import sk.ainet.lang.nn.optim.sgd
import sk.ainet.lang.tensor.tanh
import sk.ainet.lang.types.FP32
import kotlin.math.PI
import kotlin.math.cos
import kotlin.math.sin
import kotlin.math.sqrt
import kotlin.random.Random

val seed = 1337
val nSamples = 100
val noise = 0.1f
val epochs = 200
val lr = 0.1

## Data: two-moons + z-score normalisation

Deterministic port of `sklearn.datasets.make_moons` (the original micrograd demo uses it). Each sample is a `MoonPoint(x, y, label)` record; labels are in `{-1, +1}` so the final `tanh` output in `(-1, +1)` can be trained directly against them with MSE — functionally equivalent to micrograd's max-margin loss when the head passes through `tanh`.

The z-score pre-processing is plain Kotlin list operations: fit `(mean, std)` on the training split, then `map` the same statistics across both splits. SKaiNET's `skainet-data-transform` `pipeline()` builder is intentionally not used here — it is image/channel-oriented (`Normalize`, `Rescale`, `Clamp`, `Reshape`, ImageNet/MNIST presets) and not the right tool for a 100-row 2-feature in-memory dataset.

In [ ]:
data class MoonPoint(val x: Float, val y: Float, val label: Float)

fun gaussian(rng: Random): Float {
    val u1 = rng.nextFloat().coerceAtLeast(1e-7f)
    val u2 = rng.nextFloat()
    return (sqrt(-2.0 * kotlin.math.ln(u1.toDouble())) * cos(2.0 * PI * u2)).toFloat()
}

fun makeMoons(n: Int, noise: Float, rng: Random): List<MoonPoint> {
    val nOuter = n / 2
    val nInner = n - nOuter
    val outer = (0 until nOuter).map { i ->
        val t = (i.toFloat() / nOuter) * PI.toFloat()
        MoonPoint(
            x = cos(t) + gaussian(rng) * noise,
            y = sin(t) + gaussian(rng) * noise,
            label = +1f,
        )
    }
    val inner = (0 until nInner).map { i ->
        val t = (i.toFloat() / nInner) * PI.toFloat()
        MoonPoint(
            x = 1f - cos(t) + gaussian(rng) * noise,
            y = 0.5f - sin(t) + gaussian(rng) * noise,
            label = -1f,
        )
    }
    return (outer + inner).shuffled(rng)
}

data class FeatureStats(val meanX: Float, val meanY: Float, val stdX: Float, val stdY: Float) {
    fun normalize(p: MoonPoint): MoonPoint = p.copy(
        x = (p.x - meanX) / if (stdX == 0f) 1f else stdX,
        y = (p.y - meanY) / if (stdY == 0f) 1f else stdY,
    )
}

fun List<MoonPoint>.fitStats(): FeatureStats {
    val mx = (sumOf { it.x.toDouble() } / size).toFloat()
    val my = (sumOf { it.y.toDouble() } / size).toFloat()
    val sx = sqrt(sumOf { val d = it.x - mx; (d * d).toDouble() } / size).toFloat()
    val sy = sqrt(sumOf { val d = it.y - my; (d * d).toDouble() } / size).toFloat()
    return FeatureStats(mx, my, sx, sy)
}

val rng = Random(seed)
val splitIdx = (nSamples * 0.8f).toInt()
val (rawTrain, rawEval) = makeMoons(nSamples, noise, rng).let { it.take(splitIdx) to it.drop(splitIdx) }
val stats = rawTrain.fitStats()
val train = rawTrain.map(stats::normalize)
val eval = rawEval.map(stats::normalize)

println("train=${train.size} samples, eval=${eval.size} samples")
println("feature mean (pre-norm) = [${stats.meanX}, ${stats.meanY}], std = [${stats.stdX}, ${stats.stdY}]")

## Model: MLP `[2, 16, 16, 1]` with tanh activations

The Karpathy net. Tanh on the final layer keeps outputs in `(-1, +1)` for the ±1 targets.

Built on a `DefaultGraphExecutionContext` in `Phase.TRAIN` so the forward pass records a gradient tape — required for the training DSL's autograd backward step.

In [ ]:
val baseCtx = DirectCpuExecutionContext()
val trainCtx = DefaultGraphExecutionContext(
    baseOps = baseCtx.ops,
    phase = Phase.TRAIN,
    createTapeFactory = { _ -> DefaultGradientTape() }
)
val initRng = Random(seed)
val model = sequential<FP32, Float>(trainCtx) {
    input(2)
    dense(16) { weights { randn(std = 0.3f, random = initRng) } }
    activation { it.tanh() }
    dense(16) { weights { randn(std = 0.3f, random = initRng) } }
    activation { it.tanh() }
    dense(1) { weights { randn(std = 0.3f, random = initRng) } }
    activation { it.tanh() }
}
println("model trainable parameters: ${model.trainableParameters().size}")

## Training: 200 epochs of SGD with MSE

`training { ... }` returns a `TrainingRunner`. Each `runner.step(ctx, x, y)` call runs one forward pass, computes the loss, backpropagates through the tape, and applies the optimizer step + zero-grad — the canonical SKaiNET training primitive.

In [ ]:
val xTrain = data<FP32, Float>(baseCtx) {
    tensor {
        shape(train.size, 2) {
            fromList(train.flatMap { listOf(it.x, it.y) })
        }
    }
}
val yTrain = data<FP32, Float>(baseCtx) {
    tensor {
        shape(train.size, 1) {
            fromList(train.map { it.label })
        }
    }
}

val runner = training<FP32, Float> {
    model { model }
    loss { MSELoss() }
    optimizer {
        sgd(lr = lr).apply {
            model.trainableParameters().forEach { addParameter(it) }
        }
    }
}

var firstLoss = 0f
var lastLoss = 0f
repeat(epochs) { epoch ->
    val l = runner.step(trainCtx, xTrain, yTrain).data.get() as Float
    if (epoch == 0) firstLoss = l
    lastLoss = l
    if ((epoch + 1) % 20 == 0) println("epoch ${epoch + 1}/$epochs  loss=$l")
}
println("loss drop: $firstLoss → $lastLoss")

## Held-out accuracy

Forward the eval split on a fresh `DirectCpuExecutionContext` (no tape, no recording) and threshold the tanh output at `0` to get the predicted label.

In [ ]:
val evalCtx = DirectCpuExecutionContext()
val xEval = data<FP32, Float>(evalCtx) {
    tensor {
        shape(eval.size, 2) {
            fromList(eval.flatMap { listOf(it.x, it.y) })
        }
    }
}
val preds = model.forward(xEval, evalCtx)
val correct = eval.withIndex().count { (i, p) ->
    val score = preds.data.get(i, 0) as Float
    val pred = if (score >= 0f) 1f else -1f
    pred == p.label
}
val accuracy = correct.toFloat() / eval.size
println("held-out accuracy: $accuracy ($correct/${eval.size})")

## Decision boundary visualisation

Karpathy's classic plot from `demo.ipynb`: rasterise the trained model over a regular meshgrid that spans the (normalised) feature space, threshold the `tanh` output at `0` to recover the predicted class, and overlay every data point coloured by its true label.

The boundary is the contour where the network's score crosses zero — i.e. where the orange and purple tiles meet. Misclassified points show up as a coloured dot inside the wrong-coloured region.

In [ ]:
val padding = 0.5f
val step = 0.1f

val allPoints = train + eval

var xLo = Float.POSITIVE_INFINITY
var xHi = Float.NEGATIVE_INFINITY
var yLo = Float.POSITIVE_INFINITY
var yHi = Float.NEGATIVE_INFINITY
for (p in allPoints) {
    if (p.x < xLo) xLo = p.x
    if (p.x > xHi) xHi = p.x
    if (p.y < yLo) yLo = p.y
    if (p.y > yHi) yHi = p.y
}
xLo -= padding; xHi += padding
yLo -= padding; yHi += padding

val xs = generateSequence(xLo) { it + step }.takeWhile { it <= xHi }.toList()
val ys = generateSequence(yLo) { it + step }.takeWhile { it <= yHi }.toList()
val grid = ys.flatMap { yv -> xs.map { xv -> xv to yv } }

val plotCtx = DirectCpuExecutionContext()
val xMesh = data<FP32, Float>(plotCtx) {
    tensor {
        shape(grid.size, 2) {
            fromList(grid.flatMap { (gx, gy) -> listOf(gx, gy) })
        }
    }
}
val meshPreds = model.forward(xMesh, plotCtx)
val gridClass = List(grid.size) { i ->
    if ((meshPreds.data.get(i, 0) as Float) >= 0f) "+1" else "-1"
}

plot {
    tiles {
        x(grid.map { it.first.toDouble() })
        y(grid.map { it.second.toDouble() })
        fillColor(gridClass) {
            scale = categorical("+1" to Color.ORANGE, "-1" to Color.PURPLE)
        }
        width = step.toDouble() * 1.02
        height = step.toDouble() * 1.02
        alpha = 0.35
    }
    points {
        x(allPoints.map { it.x.toDouble() })
        y(allPoints.map { it.y.toDouble() })
        color(allPoints.map { if (it.label > 0f) "+1" else "-1" }) {
            scale = categorical("+1" to Color.ORANGE, "-1" to Color.PURPLE)
        }
        size = 4.0
    }
    layout {
        title = "Karpathy decision boundary on the two-moons set (normalised features)"
    }
}